# 06 · Unknown blinking：对 marginal likelihood 求导
只估计 on-state brightness a，off 背景=1，A 与 pi 已知，T=6。用可微前向算法积分全部 $2^T$ 条路径。比较 observed-data Fisher 与 complete-data Fisher：后者额外观测到 z，通常信息更多。
MC 成本小，后续位置参数也能用相同方法，只需更换 log emission。


In [ ]:
import torch, math
from torch.func import grad, vmap
torch.set_num_threads(2); torch.set_default_dtype(torch.float64); torch.manual_seed(9)
A=torch.tensor([[.85,.15],[.25,.75]]); pi=torch.tensor([.7,.3]); T=6
def marginal_loglik(a,y):
    rates=torch.stack([a*0+1,a+1])
    logb=y[:,None]*rates.log()-rates-torch.lgamma(y[:,None]+1)
    alpha=pi.log()+logb[0]
    for t in range(1,T):
        alpha=logb[t]+torch.logsumexp(alpha[:,None]+A.log(),dim=0)
    return torch.logsumexp(alpha,dim=0)
a=torch.tensor(4.)
with torch.no_grad():
    M=2000; z=torch.empty((M,T),dtype=torch.long)
    z[:,0]=torch.multinomial(pi,M,replacement=True)
    for t in range(1,T): z[:,t]=torch.multinomial(A[z[:,t-1]],1).squeeze(1)
    y=torch.poisson(1+a*z)
s=vmap(grad(marginal_loglik),in_dims=(None,0))(a,y)
f=s.square().mean()
complete_score=(z*(y/(1+a*z)-1)).sum(-1)
fc=complete_score.square().mean()
fc_exact=z.double().sum(-1).mean()/(1+a)
print('observed F, complete MC F, conditional analytic average:',f.item(),fc.item(),fc_exact.item())
print('observed a SD bound:',f.rsqrt().item())
print('observed MC SE:',s.square().std().item()/math.sqrt(M))


In [ ]:
# Independent enumeration check, including gradient.
from itertools import product
paths=torch.tensor(list(product([0,1],repeat=T)))
def enumerated(a,y):
    r=1+a*paths
    lp=pi[paths[:,0]].log()
    for t in range(1,T): lp=lp+A[paths[:,t-1],paths[:,t]].log()
    lp=lp+(y*r.log()-r-torch.lgamma(y+1)).sum(-1)
    return torch.logsumexp(lp,0)
assert torch.allclose(marginal_loglik(a,y[0]),enumerated(a,y[0]))
assert torch.allclose(grad(marginal_loglik)(a,y[0]),grad(enumerated)(a,y[0]))
print('Likelihood and score agree with path enumeration.')


不要用有限 MC 单次比较硬性断言 observed F < complete F；估计噪声可改变排序。增加样本并报告 SE。

下一步：让 transition 参数也未知，计算完整 FIM；解释固定 A 的界为什么乐观。未知 emitter 数目是 model selection，不是直接给固定维度 CRLB 多添一行。标签置换和完全重合可造成不可辨识性。
